# Combined Actuarial Analysis: Triangle and Tariff

This notebook runs the full end-to-end actuarial workflow, connecting the pricing and reserving analyses into a single coherent report.

**The core insight**: pricing and reserving are two views of the same underlying loss process. The expected loss ratio from pricing should be consistent with the loss ratio implied by reserves.


## 1. The Actuarial Workflow

```
PRICING                          RESERVING
────────                         ─────────
freMTPL data                     CAS Schedule P
    │                                │
    ▼                                ▼
Poisson GLM (frequency)          Loss Triangle
Gamma GLM (severity)             Chain Ladder → LDFs → CDFs
    │                            BF → a priori ELR blend
    ▼                                │
Pure Premium by segment          IBNR by accident year
    │                                │
    └────────────┬───────────────────┘
                 ▼
         Actuarial Report
         - Indicated rate change
         - Reserve adequacy
         - Risk segmentation
```

### The Connection

The **ELR** (expected loss ratio) is the bridge between pricing and reserving:
- In **pricing**: ELR = pure premium / gross premium. Pricing sets what the ELR *should* be.
- In **BF reserving**: ELR is the *a priori* assumption used to stabilize immature accident years.

Consistency check: the BF ELR should be close to the pricing-implied ELR. A large divergence signals either mispricing or adverse loss development.


## 2. Run Full Report

In [ ]:
import sys
sys.path.insert(0, 'actuarial')

from report import ActuarialReport

report = ActuarialReport(
    triangle_path='data/cas_triangles/ppauto_pos98-07.csv',
    freq_path='data/fremtpl/freMTPL2freq.csv',
    sev_path='data/fremtpl/freMTPL2sev.csv',
    valuation_lag=6,
)
report.run()

In [ ]:
report.summary()

In [ ]:
fig = report.plot(save_path='actuarial_report.png')
import matplotlib.pyplot as plt
plt.show()

## 3. Interpreting the Combined Report

### Reserving Side
- **Total IBNR**: BF and chain ladder broadly agree (~$258K difference, <2.1%)
- **Credibility**: most accident years have high credibility (z > 0.85) — the only meaningful BF adjustment is for the most immature year (lag 1, z=0.45)
- **Loss ratio**: 75.6-75.9% across both methods — stable and consistent

### Pricing Side
- **Mean pure premium**: $246/year across the French portfolio
- **Key risk driver**: 18-25 year olds at 3.4x average, driven by both frequency AND severity

### Rate Change Indication
- **Indicated: -5.1%** (current loss ratio 75.9% vs 80% target at 25% expense ratio)
- This means the book is slightly over-priced at current rates — a modest rate decrease is indicated
- In practice, this would be reviewed against trend factors, competitive position, and state regulations

### The Pricing-Reserving Link
- BF ELR of 75.6% is used as the a priori in reserves
- If pricing indicated a materially different ELR (e.g., 65%), it would signal that:
  - Either current reserves are overstated (assuming pricing is right)
  - Or rates need to increase to match reserve-implied loss costs (assuming reserving is right)
- The actuarial report flags this consistency check automatically


## 4. Potential Extensions

### Reserving
- **Mack (1993) variance formula**: analytical standard error around chain ladder IBNR
- **Bootstrap reserve distribution**: stochastic reserving for capital modeling
- **Clark LDF method**: parametric development curves (loglogistic, Weibull) instead of discrete LDFs — more stable with sparse data

### Pricing
- **Tweedie GLM**: models frequency × severity jointly as a compound Poisson-Gamma
- **Over-dispersed Poisson**: relaxes mean=variance assumption; more appropriate for heterogeneous portfolios
- **Interaction terms**: BonusMalus × DrivAge interaction — do young drivers with high BM score have even higher risk than the main effects would suggest?
- **XGBoost pure premium**: compare GLM relativities to gradient boosting to identify non-linear relationships the GLM misses

### Combined
- **Loss trend factors**: adjust historical losses for inflation before fitting development factors
- **Credibility-weighted ELR**: instead of using a single ELR for all accident years, weight by premium volume — more credibility to larger years
